# LeetCode 1634 — Add Two Polynomials Represented as Linked Lists bằng Python

Link đề: https://leetcode.com/problems/add-two-polynomials-represented-as-linked-lists/ · Độ khó: **Medium** · Chủ đề: Linked List, Math, Two Pointers

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for polynomial singly-linked list.
# class PolyNode:
#     def __init__(self, x=0, y=0, next=None):
#         self.coefficient = x
#         self.power = y
#         self.next = next

class Solution:
    def addPoly(self, poly1: 'PolyNode', poly2: 'PolyNode') -> 'PolyNode':
        ...
```

Notebook đi từ cách **dùng từ điển gom hệ số** (dễ nghĩ nhất) tới kỹ thuật **trộn hai danh sách đã sắp xếp** (merge) bằng hai con trỏ — cùng ý tưởng với bài "Merge Two Sorted Lists" nhưng có thêm thao tác **cộng hệ số** và **bỏ hạng tử bằng 0**.

## 1. Mục tiêu học

1. Hiểu cách biểu diễn một **đa thức** bằng danh sách liên kết `PolyNode(coefficient, power, next)` ở **dạng chuẩn**: lũy thừa giảm **nghiêm ngặt**, không có hệ số `0`.
2. Viết được cách baseline: chuyển danh sách → `dict {power: coefficient}`, cộng, sắp xếp lại, dựng danh sách mới.
3. Nắm vững kỹ thuật **merge hai danh sách đã sắp xếp** bằng hai con trỏ và **nút giả (dummy node)**, kèm 3 nhánh `>`, `<`, `==`.
4. Biết xử lý các chi tiết dễ sai: hạng tử có tổng hệ số bằng `0` phải **bị bỏ**, phần **đuôi** còn lại phải nối vào, kết quả rỗng trả về `None`.
5. Tự kiểm thử bằng sinh đa thức ngẫu nhiên có seed cố định, đối chiếu nhiều cách và kiểm tra **dạng chuẩn** của kết quả.

## 2. Đề bài & ví dụ

Một **danh sách liên kết đa thức** là danh sách liên kết đơn mà mỗi nút biểu diễn **một hạng tử** của đa thức. Mỗi nút có 3 thuộc tính:

- `coefficient`: hệ số (số nguyên). Hạng tử `9x⁴` có hệ số `9`.
- `power`: số mũ. Hạng tử `9x⁴` có số mũ `4`.
- `next`: con trỏ tới nút tiếp theo, hoặc `None` nếu là nút cuối.

Ví dụ đa thức `5x³ + 4x − 7` được biểu diễn là `5x³ → 4x¹ → (−7)x⁰`.

Danh sách phải ở **dạng chuẩn**: các nút sắp theo `power` **giảm nghiêm ngặt**, và **không có** hạng tử nào có hệ số `0`.

Cho đầu của hai danh sách `poly1` và `poly2`, hãy **cộng** hai đa thức và trả về **đầu danh sách** biểu diễn tổng (cũng ở dạng chuẩn).

**Định dạng vào/ra:** danh sách `n` nút được viết dưới dạng list các cặp `[coefficient, power]`. Ví dụ `5x³ + 4x − 7` → `[[5,3],[4,1],[-7,0]]`.

**Ví dụ 1:** `poly1 = [[1,1]]`, `poly2 = [[1,0]]` → `[[1,1],[1,0]]` (tức `x + 1`).

**Ví dụ 2:** `poly1 = [[2,2],[4,1],[3,0]]`, `poly2 = [[3,2],[-4,1],[-1,0]]` → `[[5,2],[2,0]]`.
`(2x² + 4x + 3) + (3x² − 4x − 1) = 5x² + 0x + 2`, hạng tử `0x` bị bỏ.

**Ví dụ 3:** `poly1 = [[1,2]]`, `poly2 = [[-1,2]]` → `[]` (tổng bằng 0 → danh sách rỗng).

**Ràng buộc:**
- `0 ≤ n ≤ 10⁴` (số nút của mỗi danh sách)
- `-10⁹ ≤ PolyNode.coefficient ≤ 10⁹`
- `PolyNode.coefficient != 0`
- `0 ≤ PolyNode.power ≤ 10⁹`
- `PolyNode.power > PolyNode.next.power`

Trước hết ta tự định nghĩa lớp `PolyNode` giống LeetCode và hai hàm tiện ích: dựng danh sách từ list cặp, và xuất danh sách ra list cặp.

In [1]:
import random
import time
from typing import Optional, List

class PolyNode:
    def __init__(self, x=0, y=0, next=None):
        self.coefficient = x
        self.power = y
        self.next = next

    def __repr__(self):
        return f"PolyNode({self.coefficient}, {self.power})"

def build_poly(pairs):
    """[[c, p], ...] -> đầu danh sách PolyNode (None nếu rỗng)."""
    dummy = PolyNode()
    cur = dummy
    for c, p in pairs:
        cur.next = PolyNode(c, p)
        cur = cur.next
    return dummy.next

def to_pairs(head):
    """Đầu danh sách -> [[c, p], ...]."""
    out = []
    while head:
        out.append([head.coefficient, head.power])
        head = head.next
    return out

def poly_str(pairs):
    """In đa thức cho dễ đọc: [[5,3],[4,1],[-7,0]] -> 5x^3 + 4x - 7."""
    if not pairs:
        return "0"
    parts = []
    for i, (c, p) in enumerate(pairs):
        term = "x" if p == 1 else ("" if p == 0 else f"x^{p}")
        mag = abs(c)
        body = (str(mag) if (mag != 1 or p == 0) else "") + term
        sign = "-" if c < 0 else "+"
        parts.append(("-" + body) if i == 0 and c < 0 else (body if i == 0 else f" {sign} {body}"))
    return "".join(parts)

examples = [
    ([[1, 1]], [[1, 0]], [[1, 1], [1, 0]]),
    ([[2, 2], [4, 1], [3, 0]], [[3, 2], [-4, 1], [-1, 0]], [[5, 2], [2, 0]]),
    ([[1, 2]], [[-1, 2]], []),
]
for a, b, exp in examples:
    print(f"({poly_str(a)}) + ({poly_str(b)})  ->  kỳ vọng {exp}  = {poly_str(exp)}")
print("Khứ hồi build/to_pairs:", to_pairs(build_poly([[5, 3], [4, 1], [-7, 0]])))

(x) + (1)  ->  kỳ vọng [[1, 1], [1, 0]]  = x + 1
(2x^2 + 4x + 3) + (3x^2 - 4x - 1)  ->  kỳ vọng [[5, 2], [2, 0]]  = 5x^2 + 2
(x^2) + (-x^2)  ->  kỳ vọng []  = 0
Khứ hồi build/to_pairs: [[5, 3], [4, 1], [-7, 0]]


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn và việc "đi dọc" danh sách

Mỗi nút giữ dữ liệu và con trỏ `next`. Muốn duyệt, ta giữ một biến `cur` và lặp `cur = cur.next` cho tới khi `cur is None`. Khác với list Python, ta **không** truy cập ngẫu nhiên `a[i]` được — chỉ đi tuần tự.

In [2]:
head = build_poly([[5, 3], [4, 1], [-7, 0]])
cur = head
step = 0
while cur:
    print(f"bước {step}: nút {cur}, next = {cur.next}")
    cur = cur.next
    step += 1
print("Đa thức:", poly_str(to_pairs(head)))

bước 0: nút PolyNode(5, 3), next = PolyNode(4, 1)
bước 1: nút PolyNode(4, 1), next = PolyNode(-7, 0)
bước 2: nút PolyNode(-7, 0), next = None
Đa thức: 5x^3 + 4x - 7


### 3.2 Nút giả (dummy node)

Khi dựng danh sách kết quả, nút đầu tiên là "đặc biệt" vì chưa có nút nào trước nó. Mẹo: tạo một nút giả `dummy`, luôn nối vào `tail.next`, cuối cùng trả `dummy.next`. Nhờ vậy **không cần `if head is None`** ở mỗi lần thêm. Nếu không thêm gì, `dummy.next` là `None` — đúng với "đa thức 0".

In [3]:
dummy = PolyNode()          # hệ số/số mũ của dummy không quan trọng
tail = dummy
for c, p in [[3, 4], [1, 2]]:
    tail.next = PolyNode(c, p)
    tail = tail.next
print("Kết quả:", to_pairs(dummy.next))

dummy2 = PolyNode()
print("Không thêm gì -> dummy.next =", dummy2.next)

Kết quả: [[3, 4], [1, 2]]
Không thêm gì -> dummy.next = None


### 3.3 Từ điển `dict` và `collections.defaultdict`

Để gom hệ số theo số mũ: `d[p] = d.get(p, 0) + c`. `defaultdict(int)` tự tạo giá trị `0` cho khóa mới nên viết gọn `d[p] += c`.

In [4]:
from collections import defaultdict
d = defaultdict(int)
for c, p in [[2, 2], [4, 1], [3, 0], [3, 2], [-4, 1], [-1, 0]]:
    d[p] += c
print(dict(d))
print("Bỏ hệ số 0:", {p: c for p, c in d.items() if c != 0})

{2: 5, 1: 0, 0: 2}
Bỏ hệ số 0: {2: 5, 0: 2}


### 3.4 Sắp xếp giảm dần với `sorted(..., reverse=True)`

Dạng chuẩn yêu cầu số mũ giảm dần. `sorted(d.items(), reverse=True)` sắp các cặp `(power, coef)` theo `power` giảm dần (vì so sánh tuple theo phần tử đầu trước).

In [5]:
items = {0: 2, 2: 5, 1: 0}
print(sorted(items.items(), reverse=True))
print([[c, p] for p, c in sorted(items.items(), reverse=True) if c != 0])

[(2, 5), (1, 0), (0, 2)]
[[5, 2], [2, 0]]


### 3.5 Toán tử walrus `:=` và giá trị "truthy"

`if (c := a + b):` vừa gán `c` vừa kiểm tra `c != 0` (số `0` là "falsy"). Tiện nhưng nên hiểu rõ: với số nguyên, `if c:` tương đương `if c != 0:`.

Cũng vậy, `a or b` trả `a` nếu `a` "truthy" (một nút khác `None`), ngược lại trả `b` — dùng để nối phần đuôi còn lại: `tail.next = p1 or p2`.

In [6]:
for a, b in [(3, -3), (3, 4)]:
    if (c := a + b):
        print(f"{a}+{b} = {c} -> giữ hạng tử")
    else:
        print(f"{a}+{b} = {c} -> bỏ hạng tử")

n1 = None
n2 = PolyNode(7, 0)
print("None or n2 ->", n1 or n2)
print("n2 or None ->", n2 or n1)
print("None or None ->", None or None)

3+-3 = 0 -> bỏ hạng tử
3+4 = 7 -> giữ hạng tử
None or n2 -> PolyNode(7, 0)
n2 or None -> PolyNode(7, 0)
None or None -> None


## 4. Ý tưởng nền tảng / trực giác

Cộng đa thức bằng tay: ta **ghép các hạng tử cùng số mũ** rồi cộng hệ số. Hãy làm ví dụ 2:

| Số mũ | Hệ số trong `poly1` | Hệ số trong `poly2` | Tổng | Giữ? |
|---|---|---|---|---|
| 2 | 2 | 3 | 5 | có |
| 1 | 4 | −4 | 0 | **bỏ** |
| 0 | 3 | −1 | 2 | có |

→ `5x² + 2` = `[[5,2],[2,0]]`.

Quan sát then chốt: **cả hai danh sách đều đã sắp giảm dần theo số mũ**. Vì vậy khi đọc từ trái sang phải, ta luôn biết hạng tử nào có số mũ **lớn nhất còn lại** — nó nằm ở đầu một trong hai danh sách. Đây chính là tình huống của bài trộn hai danh sách đã sắp xếp.

Thử thêm một ví dụ không cùng tập số mũ: `(x⁵ + 3x²) + (2x³ + x² − 4)`.
- Đầu hai danh sách: `x⁵` vs `2x³` → số mũ 5 lớn hơn → lấy `x⁵`.
- `3x²` vs `2x³` → lấy `2x³`.
- `3x²` vs `x²` → bằng nhau → cộng ra `4x²`, cả hai cùng tiến.
- `poly1` hết, `poly2` còn `−4` → nối thẳng vào.
→ `x⁵ + 2x³ + 4x² − 4`.

In [7]:
p1 = [[1, 5], [3, 2]]
p2 = [[2, 3], [1, 2], [-4, 0]]
i = j = 0
res = []
while i < len(p1) and j < len(p2):
    (c1, e1), (c2, e2) = p1[i], p2[j]
    if e1 > e2:
        print(f"so {c1}x^{e1} với {c2}x^{e2}: lấy {c1}x^{e1}")
        res.append([c1, e1]); i += 1
    elif e1 < e2:
        print(f"so {c1}x^{e1} với {c2}x^{e2}: lấy {c2}x^{e2}")
        res.append([c2, e2]); j += 1
    else:
        print(f"so {c1}x^{e1} với {c2}x^{e2}: cùng mũ, tổng {c1 + c2}")
        if c1 + c2:
            res.append([c1 + c2, e1])
        i += 1; j += 1
res += p1[i:] + p2[j:]
print("Kết quả:", res, "=", poly_str(res))

so 1x^5 với 2x^3: lấy 1x^5
so 3x^2 với 2x^3: lấy 2x^3
so 3x^2 với 1x^2: cùng mũ, tổng 4
Kết quả: [[1, 5], [2, 3], [4, 2], [-4, 0]] = x^5 + 2x^3 + 4x^2 - 4


## 5. Cách 1 — Brute Force / baseline: gom bằng từ điển rồi sắp xếp

**Ý tưởng:** quên đi cấu trúc "đã sắp xếp". Đọc hết hai danh sách vào một `dict` `{power: tổng hệ số}`, bỏ các khóa có tổng `0`, sắp xếp số mũ giảm dần và dựng danh sách mới.

Cách này **luôn đúng** và dễ viết, nên rất thích hợp để làm **đáp án đối chiếu** khi test.

In [8]:
from collections import defaultdict

def add_poly_dict(poly1, poly2):
    acc = defaultdict(int)
    for head in (poly1, poly2):
        while head:
            acc[head.power] += head.coefficient
            head = head.next
    dummy = tail = PolyNode()
    for p in sorted(acc, reverse=True):
        if acc[p] != 0:
            tail.next = PolyNode(acc[p], p)
            tail = tail.next
    return dummy.next

for a, b, exp in examples:
    got = to_pairs(add_poly_dict(build_poly(a), build_poly(b)))
    print(a, "+", b, "->", got, "| đúng" if got == exp else "| SAI")

[[1, 1]] + [[1, 0]] -> [[1, 1], [1, 0]] | đúng
[[2, 2], [4, 1], [3, 0]] + [[3, 2], [-4, 1], [-1, 0]] -> [[5, 2], [2, 0]] | đúng
[[1, 2]] + [[-1, 2]] -> [] | đúng


**Độ phức tạp** (gọi `n`, `m` là số nút của hai danh sách, `k ≤ n + m` là số số mũ phân biệt):
- Thời gian: `O((n + m) + k log k)` — bước sắp xếp chiếm ưu thế.
- Bộ nhớ: `O(k)` cho từ điển và danh sách mới.

**Nút thắt:** ta đã **phí** thông tin "hai danh sách đã sắp sẵn". Việc sắp xếp lại là thừa; ngoài ra phải tạo dict và nhiều nút mới. Với `n, m ≤ 10⁴` cách này vẫn chạy nhanh, nhưng trong phỏng vấn người ta mong lời giải tuyến tính `O(n + m)` không cần sắp xếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** hai danh sách đều giảm nghiêm ngặt theo `power` ⇒ hạng tử có số mũ lớn nhất **chưa xử lý** luôn nằm ở `poly1` hoặc `poly2` (đầu hiện tại). So sánh hai đầu:

- **Nếu** `poly1.power > poly2.power` **thì** hạng tử của `poly1` không thể gặp hạng tử cùng mũ trong phần còn lại của `poly2` (vì mọi số mũ còn lại của `poly2` đều ≤ `poly2.power` < `poly1.power`) ⇒ đưa nó thẳng vào kết quả, tiến `poly1`.
- **Nếu** `poly1.power < poly2.power` **thì** đối xứng: đưa `poly2` vào, tiến `poly2`.
- **Nếu** bằng nhau **thì** cộng hệ số `c`; **nếu** `c != 0` **thì** thêm nút `(c, power)`, ngược lại **bỏ qua**. Tiến **cả hai** con trỏ.
- Khi một danh sách hết, phần còn lại của danh sách kia đã ở dạng chuẩn và có mọi số mũ nhỏ hơn hạng tử cuối đã ghi ⇒ **nối nguyên đuôi**.

**Bất biến (invariant):** sau mỗi vòng lặp, danh sách kết quả là dạng chuẩn của tổng **mọi hạng tử có số mũ > max(poly1.power, poly2.power)** (số mũ của hai đầu hiện tại), và mọi số mũ trong kết quả đều lớn hơn số mũ của hai đầu còn lại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Merge hai con trỏ, tạo nút mới (không đụng vào input)

Ta đi song song hai danh sách, mỗi lần **tạo một nút mới** cho kết quả. Ưu điểm: không thay đổi `poly1`, `poly2` (an toàn nếu nơi gọi còn dùng lại chúng).

Từng đoạn:
1. `dummy = tail = PolyNode()` — nút giả.
2. Vòng `while p1 and p2`: so sánh `p1.power` và `p2.power`, xử lý 3 nhánh như mục 6.
3. Sau vòng lặp, sao chép (tạo nút mới) phần còn lại của danh sách chưa hết.

In [9]:
def add_poly_copy(poly1, poly2):
    dummy = tail = PolyNode()
    p1, p2 = poly1, poly2
    while p1 and p2:
        if p1.power > p2.power:
            tail.next = PolyNode(p1.coefficient, p1.power)
            tail = tail.next
            p1 = p1.next
        elif p1.power < p2.power:
            tail.next = PolyNode(p2.coefficient, p2.power)
            tail = tail.next
            p2 = p2.next
        else:
            c = p1.coefficient + p2.coefficient
            if c != 0:
                tail.next = PolyNode(c, p1.power)
                tail = tail.next
            p1, p2 = p1.next, p2.next
    rest = p1 or p2
    while rest:                       # sao chép đuôi
        tail.next = PolyNode(rest.coefficient, rest.power)
        tail = tail.next
        rest = rest.next
    return dummy.next

for a, b, exp in examples:
    A, B = build_poly(a), build_poly(b)
    got = to_pairs(add_poly_copy(A, B))
    print(got, "| đúng" if got == exp else "| SAI", "| input còn nguyên:", to_pairs(A) == a and to_pairs(B) == b)

[[1, 1], [1, 0]] | đúng | input còn nguyên: True
[[5, 2], [2, 0]] | đúng | input còn nguyên: True
[] | đúng | input còn nguyên: True


Code nộp LeetCode cho cách 2:

In [10]:
class SolutionCopy:
    def addPoly(self, poly1: 'PolyNode', poly2: 'PolyNode') -> 'PolyNode':
        dummy = tail = PolyNode()
        while poly1 and poly2:
            if poly1.power > poly2.power:
                tail.next = PolyNode(poly1.coefficient, poly1.power)
                poly1 = poly1.next
                tail = tail.next
            elif poly1.power < poly2.power:
                tail.next = PolyNode(poly2.coefficient, poly2.power)
                poly2 = poly2.next
                tail = tail.next
            else:
                c = poly1.coefficient + poly2.coefficient
                if c:
                    tail.next = PolyNode(c, poly1.power)
                    tail = tail.next
                poly1, poly2 = poly1.next, poly2.next
        rest = poly1 or poly2
        while rest:
            tail.next = PolyNode(rest.coefficient, rest.power)
            tail = tail.next
            rest = rest.next
        return dummy.next

print(to_pairs(SolutionCopy().addPoly(build_poly([[2, 2], [4, 1], [3, 0]]), build_poly([[3, 2], [-4, 1], [-1, 0]]))))

[[5, 2], [2, 0]]


**Độ phức tạp cách 2:** thời gian `O(n + m)` (mỗi nút được thăm đúng 1 lần), bộ nhớ phụ `O(1)` ngoài danh sách kết quả (kết quả có tối đa `n + m` nút mới).

### 7.2 Cách 3 — Merge "tái sử dụng nút" (in-place splicing)

LeetCode cho phép trả về danh sách chứa chính các nút của input. Khi một hạng tử **không có đối tác cùng mũ**, ta **móc nguyên nút đó** vào kết quả thay vì tạo nút mới. Chỉ khi hai số mũ bằng nhau ta mới cần một nút có hệ số mới (có thể tạo nút mới, hoặc sửa `coefficient` của một nút có sẵn).

Ưu điểm: không cấp phát thêm cho các hạng tử "lẻ loi"; phần đuôi được nối bằng **một phép gán** `tail.next = p1 or p2` trong `O(1)`.

Nhược điểm: danh sách input bị "xé" ra — sau lời gọi, `poly1`/`poly2` không còn là các đa thức cũ nữa.

In [11]:
def add_poly_splice(poly1, poly2):
    dummy = tail = PolyNode()
    while poly1 and poly2:
        if poly1.power > poly2.power:
            tail.next = poly1            # móc nguyên nút
            tail = poly1
            poly1 = poly1.next
        elif poly1.power < poly2.power:
            tail.next = poly2
            tail = poly2
            poly2 = poly2.next
        else:
            c = poly1.coefficient + poly2.coefficient
            nxt1, nxt2 = poly1.next, poly2.next
            if c != 0:
                poly1.coefficient = c    # dùng lại nút của poly1
                tail.next = poly1
                tail = poly1
            poly1, poly2 = nxt1, nxt2
    tail.next = poly1 or poly2          # nối đuôi O(1) — QUAN TRỌNG: cũng cắt "đuôi cũ" của tail
    return dummy.next

for a, b, exp in examples:
    got = to_pairs(add_poly_splice(build_poly(a), build_poly(b)))
    print(got, "| đúng" if got == exp else "| SAI")

[[1, 1], [1, 0]] | đúng
[[5, 2], [2, 0]] | đúng
[] | đúng


Chú ý dòng cuối `tail.next = poly1 or poly2`: khi **cả hai** đều đã hết, nó gán `tail.next = None`. Dòng này **bắt buộc**, vì `tail` có thể là một nút cũ của input mà `next` của nó vẫn trỏ tới phần đuôi cũ (ví dụ trỏ tới nút vừa bị bỏ do tổng bằng 0). Ta sẽ thấy lỗi này ở mục 13.

Code nộp LeetCode cho cách 3 (đây là cách chính, xem mục 9).

**Độ phức tạp cách 3:** thời gian `O(n + m)`, bộ nhớ phụ `O(1)` (không tạo nút mới).

### 7.3 Cách 4 — Đệ quy

Có thể viết merge theo kiểu đệ quy: "kết quả = hạng tử lớn nhất + tổng của phần còn lại".

```
add(a, b):
    nếu a rỗng: trả b;  nếu b rỗng: trả a
    nếu a.power > b.power: a.next = add(a.next, b); trả a
    nếu a.power < b.power: b.next = add(a, b.next); trả b
    c = a.coef + b.coef; rest = add(a.next, b.next)
    nếu c == 0: trả rest
    a.coefficient = c; a.next = rest; trả a
```

Rất đẹp và ngắn, **nhưng** độ sâu đệ quy có thể lên tới `n + m = 2·10⁴`, vượt giới hạn mặc định của Python (`sys.getrecursionlimit()` thường là 1000) ⇒ `RecursionError`. Chỉ nên dùng để hiểu, không nên nộp (hoặc phải tăng giới hạn đệ quy, vẫn có rủi ro tràn stack C).

In [12]:
import sys

def add_poly_rec(a, b):
    if a is None:
        return b
    if b is None:
        return a
    if a.power > b.power:
        a.next = add_poly_rec(a.next, b)
        return a
    if a.power < b.power:
        b.next = add_poly_rec(a, b.next)
        return b
    c = a.coefficient + b.coefficient
    rest = add_poly_rec(a.next, b.next)
    if c == 0:
        return rest
    a.coefficient = c
    a.next = rest
    return a

print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
for a, b, exp in examples:
    got = to_pairs(add_poly_rec(build_poly(a), build_poly(b)))
    print(got, "| đúng" if got == exp else "| SAI")

Giới hạn đệ quy hiện tại: 1000
[[1, 1], [1, 0]] | đúng
[[5, 2], [2, 0]] | đúng
[] | đúng


Thử cho thấy giới hạn của cách đệ quy (bắt ngoại lệ để notebook không lỗi):

In [13]:
big1 = build_poly([[1, 2 * k + 1] for k in range(3000, 0, -1)])   # mũ lẻ
big2 = build_poly([[1, 2 * k] for k in range(3000, 0, -1)])       # mũ chẵn
try:
    add_poly_rec(big1, big2)
    print("Chạy được (máy có giới hạn đệ quy cao)")
except RecursionError as e:
    print("RecursionError với 6000 nút:", type(e).__name__)

big1 = build_poly([[1, 2 * k + 1] for k in range(3000, 0, -1)])
big2 = build_poly([[1, 2 * k] for k in range(3000, 0, -1)])
print("Cách lặp (splice) xử lý 6000 nút:", len(to_pairs(add_poly_splice(big1, big2))), "hạng tử")

RecursionError với 6000 nút: RecursionError
Cách lặp (splice) xử lý 6000 nút: 6000 hạng tử


## 8. TRACE từng bước

Ta trace cách 3 (splice) trên ví dụ `poly1 = 2x² + 4x + 3`, `poly2 = 3x² − 4x − 1` và trên ví dụ có số mũ lệch nhau `(x⁵ + 3x²) + (2x³ + x² − 4)`.

Bảng trạng thái mong đợi cho ví dụ 2:

| Vòng | đầu `poly1` | đầu `poly2` | So sánh | Hành động | Kết quả sau vòng |
|---|---|---|---|---|---|
| 1 | `2x²` | `3x²` | `==` | tổng 5 ≠ 0 → giữ | `5x²` |
| 2 | `4x` | `−4x` | `==` | tổng 0 → **bỏ** | `5x²` |
| 3 | `3` | `−1` | `==` | tổng 2 → giữ | `5x² + 2` |
| kết thúc | `None` | `None` | — | `tail.next = None` | `5x² + 2` |

In [14]:
def trace_add_poly(pairs1, pairs2):
    poly1, poly2 = build_poly(pairs1), build_poly(pairs2)
    dummy = tail = PolyNode()
    step = 0
    print(f"Cộng ({poly_str(pairs1)}) + ({poly_str(pairs2)})")
    while poly1 and poly2:
        step += 1
        h1 = f"{poly1.coefficient}x^{poly1.power}"
        h2 = f"{poly2.coefficient}x^{poly2.power}"
        if poly1.power > poly2.power:
            action = f"mũ {poly1.power} > {poly2.power}: móc {h1}"
            tail.next = poly1; tail = poly1; poly1 = poly1.next
        elif poly1.power < poly2.power:
            action = f"mũ {poly1.power} < {poly2.power}: móc {h2}"
            tail.next = poly2; tail = poly2; poly2 = poly2.next
        else:
            c = poly1.coefficient + poly2.coefficient
            n1, n2 = poly1.next, poly2.next
            if c:
                action = f"cùng mũ {poly1.power}: tổng {c} -> giữ"
                poly1.coefficient = c; tail.next = poly1; tail = poly1
            else:
                action = f"cùng mũ {poly1.power}: tổng 0 -> BỎ"
            poly1, poly2 = n1, n2
        # in kết quả "đã chốt" (chỉ tới tail, tránh in đuôi cũ chưa cắt)
        done, cur = [], dummy.next
        while cur is not None and cur is not tail.next:
            done.append([cur.coefficient, cur.power])
            if cur is tail:
                break
            cur = cur.next
        print(f"  vòng {step}: p1={h1:>8} p2={h2:>8} | {action:<32} | kết quả = {poly_str(done)}")
    rest = poly1 or poly2
    print(f"  hết vòng lặp, nối đuôi: {to_pairs(rest)}")
    tail.next = rest
    out = to_pairs(dummy.next)
    print("  => ", out, "=", poly_str(out))
    return out

trace_add_poly([[2, 2], [4, 1], [3, 0]], [[3, 2], [-4, 1], [-1, 0]])
print()
trace_add_poly([[1, 5], [3, 2]], [[2, 3], [1, 2], [-4, 0]])

Cộng (2x^2 + 4x + 3) + (3x^2 - 4x - 1)
  vòng 1: p1=    2x^2 p2=    3x^2 | cùng mũ 2: tổng 5 -> giữ         | kết quả = 5x^2
  vòng 2: p1=    4x^1 p2=   -4x^1 | cùng mũ 1: tổng 0 -> BỎ          | kết quả = 5x^2
  vòng 3: p1=    3x^0 p2=   -1x^0 | cùng mũ 0: tổng 2 -> giữ         | kết quả = 5x^2 + 2
  hết vòng lặp, nối đuôi: []
  =>  [[5, 2], [2, 0]] = 5x^2 + 2

Cộng (x^5 + 3x^2) + (2x^3 + x^2 - 4)
  vòng 1: p1=    1x^5 p2=    2x^3 | mũ 5 > 3: móc 1x^5               | kết quả = x^5
  vòng 2: p1=    3x^2 p2=    2x^3 | mũ 2 < 3: móc 2x^3               | kết quả = x^5 + 2x^3
  vòng 3: p1=    3x^2 p2=    1x^2 | cùng mũ 2: tổng 4 -> giữ         | kết quả = x^5 + 2x^3 + 4x^2
  hết vòng lặp, nối đuôi: [[-4, 0]]
  =>  [[1, 5], [2, 3], [4, 2], [-4, 0]] = x^5 + 2x^3 + 4x^2 - 4


[[1, 5], [2, 3], [4, 2], [-4, 0]]

Trace thêm ví dụ 3 (tổng bằng 0 hoàn toàn) và trường hợp một danh sách rỗng:

In [15]:
trace_add_poly([[1, 2]], [[-1, 2]])
print()
trace_add_poly([], [[7, 3], [1, 0]])

Cộng (x^2) + (-x^2)
  vòng 1: p1=    1x^2 p2=   -1x^2 | cùng mũ 2: tổng 0 -> BỎ          | kết quả = 0
  hết vòng lặp, nối đuôi: []
  =>  [] = 0

Cộng (0) + (7x^3 + 1)
  hết vòng lặp, nối đuôi: [[7, 3], [1, 0]]
  =>  [[7, 3], [1, 0]] = 7x^3 + 1


[[7, 3], [1, 0]]

## 9. Code nộp LeetCode

Đây là lời giải chính (cách 3 — merge tái sử dụng nút). Trên LeetCode lớp `PolyNode` đã có sẵn, **không** cần định nghĩa lại.

In [16]:
# Definition for polynomial singly-linked list.
# class PolyNode:
#     def __init__(self, x=0, y=0, next=None):
#         self.coefficient = x
#         self.power = y
#         self.next = next

class Solution:
    def addPoly(self, poly1: 'PolyNode', poly2: 'PolyNode') -> 'PolyNode':
        dummy = tail = PolyNode()
        while poly1 and poly2:
            if poly1.power > poly2.power:
                tail.next = poly1
                tail = poly1
                poly1 = poly1.next
            elif poly1.power < poly2.power:
                tail.next = poly2
                tail = poly2
                poly2 = poly2.next
            else:
                c = poly1.coefficient + poly2.coefficient
                nxt1, nxt2 = poly1.next, poly2.next
                if c != 0:
                    poly1.coefficient = c
                    tail.next = poly1
                    tail = poly1
                poly1, poly2 = nxt1, nxt2
        tail.next = poly1 or poly2
        return dummy.next

for a, b, exp in examples:
    got = to_pairs(Solution().addPoly(build_poly(a), build_poly(b)))
    assert got == exp, (a, b, got, exp)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

Gọi `S` là danh sách kết quả đã dựng, `a`, `b` là đầu hiện tại của hai danh sách.

**Bất biến:** (I1) `S` là dạng chuẩn của tổng mọi hạng tử đã xử lý; (I2) mọi số mũ trong `S` **lớn hơn** mọi số mũ còn lại trong `a` và `b`; (I3) mọi hạng tử chưa xử lý nằm trong `a` hoặc `b`.

- **Khởi tạo:** `S` rỗng — đúng hiển nhiên.
- **Nhánh `a.power > b.power`:** vì `b` giảm nghiêm ngặt, mọi số mũ còn lại trong `b` ≤ `b.power` < `a.power`; trong `a` thì các số mũ sau `a` < `a.power`. ⇒ Số mũ `a.power` **xuất hiện đúng một lần** trong toàn bộ phần chưa xử lý, nên hệ số của nó trong tổng chính là `a.coefficient ≠ 0` (đề bảo đảm hệ số khác 0). Thêm vào cuối `S` giữ được (I1) (vẫn giảm nghiêm ngặt nhờ (I2)) và (I2).
- **Nhánh `<`:** đối xứng.
- **Nhánh `==`:** số mũ `e` xuất hiện đúng 2 lần (một ở mỗi danh sách), tổng hệ số là `c`. Nếu `c ≠ 0` thêm `(c, e)`, nếu `c = 0` thì theo định nghĩa dạng chuẩn hạng tử phải bị bỏ. Cả hai cách đều giữ (I1), (I2).
- **Kết thúc vòng lặp:** một danh sách rỗng, danh sách còn lại (nếu có) ở dạng chuẩn, mọi số mũ của nó nhỏ hơn mọi số mũ trong `S` (theo I2) và không có đối tác cùng mũ ⇒ nối nguyên vào cuối `S` cho dạng chuẩn của toàn bộ tổng.

**Dừng:** mỗi vòng ít nhất một con trỏ tiến 1 bước, tổng số bước ≤ `n + m`.

Riêng cách 3: việc sửa `poly1.coefficient` là an toàn vì nút đó đã "rời" khỏi phần chưa xử lý (ta lưu `nxt1` trước), và dòng `tail.next = poly1 or poly2` bảo đảm không còn liên kết cũ "lơ lửng" sau `tail`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(poly1)`, `m = len(poly2)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Dict + sort | `O((n+m) + k log k)`, `k ≤ n+m` | `O(n+m)` | sắp xếp lại là thừa |
| 2. Merge, tạo nút mới | `O(n+m)` | `O(1)` (+ `O(n+m)` cho output) | input giữ nguyên |
| 3. Merge, tái dùng nút | `O(n+m)` | `O(1)` | đuôi nối `O(1)`, input bị thay đổi |
| 4. Đệ quy | `O(n+m)` | `O(n+m)` stack | dễ `RecursionError` |

Không có khác biệt giữa trường hợp trung bình và xấu nhất cho cách 2–3: mỗi vòng lặp tiêu thụ ít nhất 1 nút. Trường hợp **tốt nhất** của cách 3 là khi một danh sách ngắn: vòng lặp chỉ chạy `≤ 2·min(n, m)` lần rồi nối đuôi trong `O(1)`.

Lưu ý giá trị: hệ số tới `±10⁹`, tổng tới `±2·10⁹` — Python dùng số nguyên vô hạn nên không lo tràn (ở C++/Java `int` 32-bit cũng vẫn vừa vì `2·10⁹ < 2³¹ − 1 ≈ 2.147·10⁹`).

In [17]:
def timed(fn, A, B):
    t = time.perf_counter(); r = fn(A, B); return r, time.perf_counter() - t

rnd = random.Random(1634)
def rand_poly(rnd, size, max_pow):
    pows = sorted(rnd.sample(range(max_pow + 1), size), reverse=True)
    return [[rnd.choice([-1, 1]) * rnd.randint(1, 10**9), p] for p in pows]

a = rand_poly(rnd, 10**4, 3 * 10**4)
b = rand_poly(rnd, 10**4, 3 * 10**4)
for name, fn in [("dict+sort", add_poly_dict), ("copy merge", add_poly_copy), ("splice merge", add_poly_splice)]:
    r, dt = timed(fn, build_poly(a), build_poly(b))
    print(f"{name:<13}: {len(to_pairs(r))} hạng tử, {dt * 1000:.1f} ms")

dict+sort    : 16676 hạng tử, 8.3 ms
copy merge   : 16676 hạng tử, 4.8 ms
splice merge : 16676 hạng tử, 1.2 ms


## 12. Bộ test

Ta kiểm tra: (1) ví dụ đề; (2) edge case: một hoặc cả hai rỗng, triệt tiêu hoàn toàn, triệt tiêu ở giữa, số mũ 0 và `10⁹`, hệ số biên `±10⁹`; (3) ngẫu nhiên có seed, đúng ràng buộc (hệ số khác 0, mũ giảm nghiêm ngặt); (4) kết quả phải ở **dạng chuẩn**.

Vì cách 3 và cách 4 **phá** input, mỗi lần gọi ta dựng danh sách mới từ list cặp.

In [18]:
def is_standard(pairs):
    """Dạng chuẩn: mũ giảm nghiêm ngặt, không có hệ số 0."""
    return all(c != 0 for c, _ in pairs) and all(pairs[i][1] > pairs[i + 1][1] for i in range(len(pairs) - 1))

def reference(a, b):
    acc = defaultdict(int)
    for c, p in a + b:
        acc[p] += c
    return [[acc[p], p] for p in sorted(acc, reverse=True) if acc[p] != 0]

ALL = {
    "dict": add_poly_dict,
    "copy": add_poly_copy,
    "splice": add_poly_splice,
    "rec": add_poly_rec,
    "Solution": lambda x, y: Solution().addPoly(x, y),
    "SolutionCopy": lambda x, y: SolutionCopy().addPoly(x, y),
}

def run_all(a, b):
    exp = reference(a, b)
    for name, fn in ALL.items():
        if name == "rec" and len(a) + len(b) > 300:
            continue                      # đệ quy chỉ test trên input nhỏ (tránh RecursionError)
        got = to_pairs(fn(build_poly(a), build_poly(b)))
        assert got == exp, (name, a, b, got, exp)
        assert is_standard(got), (name, got)
    return exp

# (1) ví dụ đề
for a, b, exp in examples:
    assert run_all(a, b) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [19]:
# (2) edge case
edge = [
    ([], [], []),
    ([], [[3, 2]], [[3, 2]]),
    ([[3, 2]], [], [[3, 2]]),
    ([[1, 2], [1, 1], [1, 0]], [[-1, 2], [-1, 1], [-1, 0]], []),            # triệt tiêu toàn bộ
    ([[1, 3], [2, 1]], [[-2, 1]], [[1, 3]]),                                # triệt tiêu hạng tử cuối
    ([[1, 3], [2, 1], [5, 0]], [[-2, 1]], [[1, 3], [5, 0]]),                # triệt tiêu ở giữa
    ([[10**9, 10**9]], [[10**9, 10**9]], [[2 * 10**9, 10**9]]),             # biên lớn
    ([[-10**9, 0]], [[10**9, 0]], []),
    ([[1, 5], [1, 3], [1, 1]], [[1, 4], [1, 2], [1, 0]], [[1, 5], [1, 4], [1, 3], [1, 2], [1, 1], [1, 0]]),  # xen kẽ
    ([[1, 10]], [[1, 2], [1, 1]], [[1, 10], [1, 2], [1, 1]]),               # không giao nhau
]
for a, b, exp in edge:
    assert run_all(a, b) == exp, (a, b)
print("Edge case:", len(edge), "bộ OK")

Edge case: 10 bộ OK


In [20]:
# (3) ngẫu nhiên, đúng ràng buộc
def rand_poly_small(rnd, max_len=8, max_pow=10, max_c=3):
    size = rnd.randint(0, min(max_len, max_pow + 1))
    pows = sorted(rnd.sample(range(max_pow + 1), size), reverse=True)
    return [[rnd.choice([-1, 1]) * rnd.randint(1, max_c), p] for p in pows]

rnd = random.Random(20240)
for _ in range(3000):
    a = rand_poly_small(rnd)
    # thường cho b là "đối" của một phần a để tạo nhiều triệt tiêu
    if rnd.random() < 0.4:
        b = [[-c, p] for c, p in a if rnd.random() < 0.6]
    else:
        b = rand_poly_small(rnd)
    run_all(a, b)
print("3000 cặp ngẫu nhiên nhỏ: OK")

for _ in range(30):
    a = rand_poly_small(rnd, max_len=600, max_pow=10**9, max_c=10**9)
    b = rand_poly_small(rnd, max_len=600, max_pow=10**9, max_c=10**9)
    run_all(a, b)
print("30 cặp lớn (mũ tới 1e9, hệ số tới 1e9): OK")

3000 cặp ngẫu nhiên nhỏ: OK


30 cặp lớn (mũ tới 1e9, hệ số tới 1e9): OK


In [21]:
# (4) Kiểm tra bằng cách "thế số": giá trị đa thức tổng tại vài điểm x phải bằng tổng giá trị
def evaluate(pairs, x, mod=10**9 + 7):
    return sum(c * pow(x, p, mod) for c, p in pairs) % mod

rnd = random.Random(7)
for _ in range(500):
    a = rand_poly_small(rnd, max_len=10, max_pow=50, max_c=100)
    b = rand_poly_small(rnd, max_len=10, max_pow=50, max_c=100)
    got = to_pairs(Solution().addPoly(build_poly(a), build_poly(b)))
    for x in (2, 3, 12345):
        assert evaluate(got, x) == (evaluate(a, x) + evaluate(b, x)) % (10**9 + 7)
print("Thế số kiểm tra: OK")

Thế số kiểm tra: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên bỏ hạng tử có tổng hệ số bằng 0

In [22]:
def add_poly_bug_zero(poly1, poly2):
    dummy = tail = PolyNode()
    while poly1 and poly2:
        if poly1.power > poly2.power:
            tail.next = PolyNode(poly1.coefficient, poly1.power); poly1 = poly1.next
        elif poly1.power < poly2.power:
            tail.next = PolyNode(poly2.coefficient, poly2.power); poly2 = poly2.next
        else:
            tail.next = PolyNode(poly1.coefficient + poly2.coefficient, poly1.power)   # SAI: không kiểm tra 0
            poly1, poly2 = poly1.next, poly2.next
        tail = tail.next
    tail.next = poly1 or poly2
    return dummy.next

a, b = [[2, 2], [4, 1], [3, 0]], [[3, 2], [-4, 1], [-1, 0]]
print("sai :", to_pairs(add_poly_bug_zero(build_poly(a), build_poly(b))), "(có [0, 1] — không phải dạng chuẩn)")
print("đúng:", to_pairs(Solution().addPoly(build_poly(a), build_poly(b))))
print("ví dụ 3 sai:", to_pairs(add_poly_bug_zero(build_poly([[1, 2]]), build_poly([[-1, 2]]))), "(phải là [])")

sai : [[5, 2], [0, 1], [2, 0]] (có [0, 1] — không phải dạng chuẩn)
đúng: [[5, 2], [2, 0]]
ví dụ 3 sai: [[0, 2]] (phải là [])


**Vì sao sai:** đề yêu cầu dạng chuẩn — hạng tử hệ số `0` **phải bị lược bỏ**. **Sửa:** chỉ nối nút khi `c != 0`, nhưng **vẫn** tiến cả hai con trỏ.

### Lỗi 2 — Quên dòng `tail.next = poly1 or poly2` khi tái sử dụng nút

Nhiều bạn nghĩ "nếu cả hai đều hết thì khỏi nối gì". Nhưng `tail` là nút **cũ** của input, `tail.next` vẫn trỏ tới nút cũ phía sau (có thể là nút vừa bị **bỏ** vì tổng bằng 0).

In [23]:
def add_poly_bug_tail(poly1, poly2):
    dummy = tail = PolyNode()
    while poly1 and poly2:
        if poly1.power > poly2.power:
            tail.next = poly1; tail = poly1; poly1 = poly1.next
        elif poly1.power < poly2.power:
            tail.next = poly2; tail = poly2; poly2 = poly2.next
        else:
            c = poly1.coefficient + poly2.coefficient
            n1, n2 = poly1.next, poly2.next
            if c:
                poly1.coefficient = c; tail.next = poly1; tail = poly1
            poly1, poly2 = n1, n2
    if poly1 or poly2:                     # SAI: bỏ qua khi cả hai đều None
        tail.next = poly1 or poly2
    return dummy.next

a, b = [[1, 3], [2, 1]], [[-2, 1]]
print("sai :", to_pairs(add_poly_bug_tail(build_poly(a), build_poly(b))), "(còn dính [2, 1] cũ)")
print("đúng:", to_pairs(Solution().addPoly(build_poly(a), build_poly(b))))

sai : [[1, 3], [2, 1]] (còn dính [2, 1] cũ)
đúng: [[1, 3]]


**Vì sao sai:** `x³` được móc vào kết quả, `x³.next` vẫn là nút `2x` của `poly1`. Cặp `2x` và `−2x` triệt tiêu nên không ai sửa `x³.next` ⇒ kết quả còn dính `2x`. **Sửa:** luôn gán `tail.next = poly1 or poly2` (kể cả khi bằng `None`).

### Lỗi 3 — Chỉ tiến một con trỏ khi hai số mũ bằng nhau

In [24]:
def add_poly_bug_advance(poly1, poly2):
    dummy = tail = PolyNode()
    while poly1 and poly2:
        if poly1.power > poly2.power:
            tail.next = PolyNode(poly1.coefficient, poly1.power); poly1 = poly1.next; tail = tail.next
        elif poly1.power < poly2.power:
            tail.next = PolyNode(poly2.coefficient, poly2.power); poly2 = poly2.next; tail = tail.next
        else:
            c = poly1.coefficient + poly2.coefficient
            if c:
                tail.next = PolyNode(c, poly1.power); tail = tail.next
            poly1 = poly1.next                 # SAI: quên poly2 = poly2.next
    rest = poly1 or poly2
    while rest:
        tail.next = PolyNode(rest.coefficient, rest.power); tail = tail.next; rest = rest.next
    return dummy.next

a, b = [[1, 1]], [[2, 1]]
print("sai :", to_pairs(add_poly_bug_advance(build_poly(a), build_poly(b))), "(2x bị cộng rồi còn bị chép lại)")
print("đúng:", to_pairs(Solution().addPoly(build_poly(a), build_poly(b))))

sai : [[3, 1], [2, 1]] (2x bị cộng rồi còn bị chép lại)
đúng: [[3, 1]]


**Vì sao sai:** hạng tử `2x` của `poly2` đã được "tiêu thụ" khi cộng, nhưng con trỏ `poly2` không tiến nên nó bị xử lý **lần nữa**. **Sửa:** `poly1, poly2 = poly1.next, poly2.next`.

### Lỗi 4 — So sánh ngược chiều (tưởng danh sách tăng dần)

In [25]:
def add_poly_bug_order(poly1, poly2):
    dummy = tail = PolyNode()
    while poly1 and poly2:
        if poly1.power < poly2.power:          # SAI: lấy mũ NHỎ hơn trước
            tail.next = PolyNode(poly1.coefficient, poly1.power); poly1 = poly1.next; tail = tail.next
        elif poly1.power > poly2.power:
            tail.next = PolyNode(poly2.coefficient, poly2.power); poly2 = poly2.next; tail = tail.next
        else:
            c = poly1.coefficient + poly2.coefficient
            if c:
                tail.next = PolyNode(c, poly1.power); tail = tail.next
            poly1, poly2 = poly1.next, poly2.next
    rest = poly1 or poly2
    while rest:
        tail.next = PolyNode(rest.coefficient, rest.power); tail = tail.next; rest = rest.next
    return dummy.next

a, b = [[1, 1]], [[1, 0]]
got = to_pairs(add_poly_bug_order(build_poly(a), build_poly(b)))
print("sai :", got, "| dạng chuẩn?", is_standard(got))
print("đúng:", to_pairs(Solution().addPoly(build_poly(a), build_poly(b))))

sai : [[1, 0], [1, 1]] | dạng chuẩn? False
đúng: [[1, 1], [1, 0]]


**Vì sao sai:** danh sách sắp **giảm dần**, phải lấy số mũ **lớn hơn** trước; lấy nhỏ trước làm kết quả không còn giảm nghiêm ngặt. **Sửa:** nhánh đầu là `poly1.power > poly2.power`.

### Lỗi 5 — Sửa hệ số tại chỗ nhưng chưa lưu `next` (cách splice)

Nếu bạn viết `tail.next = poly1; tail = poly1` rồi mới `poly1 = poly1.next` thì vẫn ổn. Nhưng nếu ở nhánh `==` bạn gán `tail.next = poly1` **và** sau đó gán `tail.next = ...` cho nút khác trước khi lưu `poly1.next`, liên kết bị mất. Quy tắc an toàn: **luôn lưu `nxt1, nxt2` trước khi thay đổi bất kỳ `next` nào**, như trong code nộp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "hai danh sách đã sắp xếp" + "gộp/cộng/hợp" ⇒ nghĩ ngay tới **merge hai con trỏ** như trong Merge Sort.
- Luôn dùng **nút giả `dummy`** khi dựng danh sách liên kết mới — bớt hẳn một nhánh `if`.
- Merge có 3 nhánh `>`, `<`, `==`; nhánh `==` là nơi chứa "logic riêng" của bài (ở đây: cộng và bỏ 0).
- Sau vòng lặp **luôn** xử lý phần đuôi: `tail.next = p1 or p2`.
- Viết sẵn `build_*` / `to_*` để test cục bộ — bài linked list trên LeetCode không cho in dễ dàng.
- Khi tái sử dụng nút input, hãy nghĩ tới các "liên kết cũ" còn sót lại; khi nghi ngờ, tạo nút mới cho an toàn.
- Trong phỏng vấn, nêu cách dict + sort trước (đúng nhưng chậm hơn) rồi tối ưu bằng việc tận dụng tính đã sắp xếp.

## 15. Biến thể của bài

1. **LeetCode 21 — Merge Two Sorted Lists:** chính là bộ khung của bài này, không có bước cộng.
2. **LeetCode 2 — Add Two Numbers:** cộng hai số lưu ngược trong danh sách liên kết (có nhớ), cũng là đi song song hai danh sách.
3. **Trừ hai đa thức:** đổi dấu `poly2` rồi cộng.
4. **Cộng `k` đa thức:** dùng heap (giống LeetCode 23 — Merge k Sorted Lists) hoặc chia để trị cộng từng cặp.
5. **Nhân hai đa thức** (dạng thưa): cộng dồn `c1·c2` vào `dict[p1 + p2]` rồi sắp xếp; hoặc nhân từng hạng tử rồi cộng dần bằng hàm `addPoly`.
6. **Đạo hàm đa thức:** `c·xᵖ → (c·p)·xᵖ⁻¹`, bỏ hạng tử hằng số.
7. **Đa thức biểu diễn dạng mảng dày** `coef[i]` là hệ số của `xⁱ`: khi số mũ nhỏ, cộng theo chỉ số; khi số mũ tới `10⁹` (như bài này) dạng **thưa** (danh sách liên kết/dict) mới khả thi.

In [26]:
def sub_poly(a_pairs, b_pairs):
    neg = [[-c, p] for c, p in b_pairs]
    return to_pairs(Solution().addPoly(build_poly(a_pairs), build_poly(neg)))

def mul_poly(a_pairs, b_pairs):
    acc = defaultdict(int)
    for c1, p1 in a_pairs:
        for c2, p2 in b_pairs:
            acc[p1 + p2] += c1 * c2
    return [[acc[p], p] for p in sorted(acc, reverse=True) if acc[p]]

print("(x^2 + 3x) - (x^2 - 1) =", poly_str(sub_poly([[1, 2], [3, 1]], [[1, 2], [-1, 0]])))
print("(x + 1)(x - 1) =", poly_str(mul_poly([[1, 1], [1, 0]], [[1, 1], [-1, 0]])))
print("(x + 1)^2 =", poly_str(mul_poly([[1, 1], [1, 0]], [[1, 1], [1, 0]])))

(x^2 + 3x) - (x^2 - 1) = 3x + 1
(x + 1)(x - 1) = x^2 - 1
(x + 1)^2 = x^2 + 2x + 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dict + sort | gom hệ số theo mũ, sắp lại | `O((n+m) log(n+m))` | `O(n+m)` | làm đáp án đối chiếu; khi input **chưa** sắp xếp |
| 2. Merge tạo nút mới | hai con trỏ, 3 nhánh, copy | `O(n+m)` | `O(1)` phụ | khi cần giữ nguyên input |
| 3. Merge tái dùng nút | hai con trỏ, móc nút cũ | `O(n+m)` | `O(1)` | nộp LeetCode — nhanh, ít cấp phát |
| 4. Đệ quy | "hạng tử lớn nhất + phần còn lại" | `O(n+m)` | `O(n+m)` stack | để hiểu; tránh với `n` lớn trong Python |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Không chạy code, kết quả của `addPoly([[3,4],[1,0]], [[-3,4],[2,2]])` là gì?

**Bài 2 (tính tay):** Trace cách merge trên `poly1 = [[1,6],[2,3],[1,0]]`, `poly2 = [[4,5],[-2,3],[-1,0]]`: ghi ra từng vòng nhánh nào được chọn và kết quả cuối.

**Bài 3 (điền TODO):** Hoàn thiện hàm `count_terms_of_sum(a_pairs, b_pairs)` trả về **số hạng tử** của tổng, **không** dựng danh sách mới (chỉ dùng hai chỉ số `i`, `j` trên list cặp).

**Bài 4 (tự viết lại):** Viết `sub_poly_merge(a_pairs, b_pairs)` tính `a − b` trực tiếp bằng merge hai chỉ số (không đổi dấu trước), trả về list cặp ở dạng chuẩn.

**Bài 5 (mini challenge):** Viết `add_k_polys(list_of_pairs)` cộng `k` đa thức (mỗi đa thức dạng chuẩn) trong `O(N log k)` với `N` là tổng số hạng tử, dùng `heapq`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_terms_of_sum(a_pairs, b_pairs):
    i = j = 0
    count = 0
    # TODO: duyệt như merge; ở nhánh cùng mũ chỉ đếm khi tổng khác 0
    # TODO: cộng thêm số hạng tử còn lại ở đuôi
    return count

# Bài 4 — TODO
def sub_poly_merge(a_pairs, b_pairs):
    # TODO: như merge, nhưng hạng tử chỉ có ở b phải đổi dấu
    pass

# Bài 5 — TODO
def add_k_polys(list_of_pairs):
    # TODO: heap chứa (-power, chỉ số đa thức, vị trí trong đa thức)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Có hạng tử nào cùng số mũ không?
- Mức 2: `3x⁴` và `−3x⁴` triệt tiêu.
- Mức 3: Còn lại `2x²` và `1`, sắp giảm dần.

**Bài 2**
- Mức 1: So sánh mũ 6 và 5 trước.
- Mức 2: Mũ 3: `2 + (−2) = 0` ⇒ bỏ. Mũ 0: `1 + (−1) = 0` ⇒ bỏ.
- Mức 3: Kết quả chỉ còn `x⁶ + 4x⁵`.

**Bài 3**
- Mức 1: Khung giống hệt merge, nhưng thay `append` bằng `count += 1`.
- Mức 2: Nhánh `==`: `count += (a[i][0] + b[j][0] != 0)`.
- Mức 3: Cuối cùng `count += (len(a) - i) + (len(b) - j)`.

**Bài 4**
- Mức 1: `a − b = a + (−b)`, nhưng ta phải "đổi dấu khi lấy".
- Mức 2: Nhánh lấy từ `b`: thêm `[-c, p]`; nhánh bằng: `c = ca - cb`.
- Mức 3: Đuôi của `b` cũng phải đổi dấu.

**Bài 5**
- Mức 1: Giống Merge k Sorted Lists, nhưng số mũ giảm dần ⇒ heap theo `-power`.
- Mức 2: Mỗi lần lấy ra **tất cả** phần tử có cùng số mũ nhỏ nhất của `-power`, cộng hệ số lại.
- Mức 3: Chỉ ghi khi tổng khác 0; sau khi lấy một phần tử, đẩy phần tử kế tiếp của đa thức đó vào heap.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ans1 = to_pairs(Solution().addPoly(build_poly([[3, 4], [1, 0]]), build_poly([[-3, 4], [2, 2]])))
print("Bài 1:", ans1)
assert ans1 == [[2, 2], [1, 0]]

# Bài 2
out2 = trace_add_poly([[1, 6], [2, 3], [1, 0]], [[4, 5], [-2, 3], [-1, 0]])
assert out2 == [[1, 6], [4, 5]]

Bài 1: [[2, 2], [1, 0]]
Cộng (x^6 + 2x^3 + 1) + (4x^5 - 2x^3 - 1)
  vòng 1: p1=    1x^6 p2=    4x^5 | mũ 6 > 5: móc 1x^6               | kết quả = x^6
  vòng 2: p1=    2x^3 p2=    4x^5 | mũ 3 < 5: móc 4x^5               | kết quả = x^6 + 4x^5
  vòng 3: p1=    2x^3 p2=   -2x^3 | cùng mũ 3: tổng 0 -> BỎ          | kết quả = x^6 + 4x^5
  vòng 4: p1=    1x^0 p2=   -1x^0 | cùng mũ 0: tổng 0 -> BỎ          | kết quả = x^6 + 4x^5
  hết vòng lặp, nối đuôi: []
  =>  [[1, 6], [4, 5]] = x^6 + 4x^5


In [29]:
# Bài 3
def count_terms_of_sum_sol(a, b):
    i = j = count = 0
    while i < len(a) and j < len(b):
        if a[i][1] > b[j][1]:
            count += 1; i += 1
        elif a[i][1] < b[j][1]:
            count += 1; j += 1
        else:
            count += (a[i][0] + b[j][0] != 0); i += 1; j += 1
    return count + (len(a) - i) + (len(b) - j)

# Bài 4
def sub_poly_merge_sol(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i][1] > b[j][1]:
            out.append(a[i][:]); i += 1
        elif a[i][1] < b[j][1]:
            out.append([-b[j][0], b[j][1]]); j += 1
        else:
            c = a[i][0] - b[j][0]
            if c:
                out.append([c, a[i][1]])
            i += 1; j += 1
    out += [x[:] for x in a[i:]] + [[-c, p] for c, p in b[j:]]
    return out

rnd = random.Random(99)
for _ in range(2000):
    a = rand_poly_small(rnd); b = rand_poly_small(rnd)
    assert count_terms_of_sum_sol(a, b) == len(reference(a, b))
    assert sub_poly_merge_sol(a, b) == reference(a, [[-c, p] for c, p in b])
print("Bài 3, Bài 4: OK")

Bài 3, Bài 4: OK


In [30]:
# Bài 5
import heapq

def add_k_polys_sol(list_of_pairs):
    heap = [(-poly[0][1], idx, 0) for idx, poly in enumerate(list_of_pairs) if poly]
    heapq.heapify(heap)
    out = []
    while heap:
        neg_p = heap[0][0]
        total = 0
        while heap and heap[0][0] == neg_p:            # gom mọi hạng tử cùng mũ
            _, idx, pos = heapq.heappop(heap)
            total += list_of_pairs[idx][pos][0]
            if pos + 1 < len(list_of_pairs[idx]):
                nxt = list_of_pairs[idx][pos + 1]
                heapq.heappush(heap, (-nxt[1], idx, pos + 1))
        if total:
            out.append([total, -neg_p])
    return out

rnd = random.Random(5)
for _ in range(800):
    polys = [rand_poly_small(rnd) for _ in range(rnd.randint(0, 6))]
    allterms = [t for p in polys for t in p]
    assert add_k_polys_sol(polys) == reference(allterms, [])
print("Bài 5: OK  | ví dụ:", poly_str(add_k_polys_sol([[[1, 2]], [[2, 2], [1, 0]], [[-3, 2], [4, 1]]])))

Bài 5: OK  | ví dụ: 4x + 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template merge hai danh sách đã sắp xếp (giảm dần) có xử lý "cùng khóa":**

```python
dummy = tail = Node()
while a and b:
    if a.key > b.key:
        tail.next = a; tail = a; a = a.next
    elif a.key < b.key:
        tail.next = b; tail = b; b = b.next
    else:
        na, nb = a.next, b.next          # lưu trước khi sửa
        v = combine(a, b)
        if keep(v):                      # bài này: v != 0
            a.val = v; tail.next = a; tail = a
        a, b = na, nb
tail.next = a or b                       # LUÔN gán, kể cả None
return dummy.next
```

**Checklist:**
- [ ] So sánh đúng chiều: danh sách **giảm dần** ⇒ lấy `power` **lớn hơn** trước?
- [ ] Nhánh bằng nhau: tiến **cả hai** con trỏ?
- [ ] Bỏ hạng tử có tổng hệ số `0`?
- [ ] Luôn có `tail.next = poly1 or poly2` sau vòng lặp (cắt liên kết cũ)?
- [ ] Kết quả rỗng trả `None` (tức `dummy.next`)?
- [ ] Không dùng đệ quy với `n + m` tới `2·10⁴` trong Python?
- [ ] Chữ ký đúng: `def addPoly(self, poly1: 'PolyNode', poly2: 'PolyNode') -> 'PolyNode'`, không định nghĩa lại `PolyNode` khi nộp?